## Step 1: Environment Setup
Before we begin, we need to install the necessary libraries. The following cells contain the installation commands for both Google Colab and Local environments. We are installing:
* **`unsloth` & `trl`**: For efficient model loading and Reinforcement Learning (GRPO) training.
* **`peft` & `bitsandbytes`**: For Parameter-Efficient Fine-Tuning (LoRA) and 4-bit quantization to save memory.
* **`datasets`**: To handle our training data.
* **`sentence-transformers`**: To run a lightweight, local embedding model for our semantic similarity reward function.
* **`wandb`**: To track and visualize our training metrics and rewards in real-time.

*Note: Uncomment the cell that corresponds to your environment.*

In [1]:
#########
# colab #
#########

# # Install Unsloth
# !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# # Force install the latest TRL version from GitHub to ensure GRPO is available
# !pip install git+https://github.com/huggingface/trl.git@main

# # Install other dependencies
# !pip install peft accelerate bitsandbytes datasets sentence-transformers wandb

In [2]:
#########
# local #
#########

# # Python 3.10
# !pip install unsloth trl peft accelerate bitsandbytes datasets sentence-transformers wandb
# !pip install mergekit
# !pip install llm_blender
# !pip install weave

### Patching Transformers Cache (Local Environment Only)
If you are running this locally, it is good practice to explicitly set the Hugging Face cache directory to avoid downloading large models into temporary or restricted folders.

In [3]:
#########
# local #
#########

# Patch transformers
import os
import transformers.utils.hub

transformers.utils.hub.TRANSFORMERS_CACHE = os.getenv("HF_HOME", os.path.expanduser("~/.cache/huggingface/hub"))

/home/andres/Documentos/Big Data/TFM/Lab/LLMs bias mitigation/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 2: Importing Libraries
Here we import all the required modules for our training pipeline. We import `FastLanguageModel` from Unsloth for highly optimized inference and training, and the `GRPOConfig` and `GRPOTrainer` from the `trl` library to manage the reinforcement learning loop.

In [4]:
import re
import wandb # type: ignore
import warnings # type: ignore
from unsloth import FastLanguageModel, PatchFastRL # type: ignore
import transformers # type: ignore
import torch # type: ignore
from datasets import Dataset # type: ignore
from trl import GRPOConfig, GRPOTrainer # type: ignore
from sentence_transformers import SentenceTransformer, util # type: ignore

# Ignoring warnings for cleaner output
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
transformers.logging.set_verbosity_error()

# Train visualization with Weights & Biases
wandb.login()

# 1. Patch Unsloth to optimize memory usage and speed for GRPO
PatchFastRL("GRPO", FastLanguageModel)

/tmp/ipykernel_7707/1288167950.py:4: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel, PatchFastRL # type: ignore


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: andreslilloortiz (andreslilloortiz-santiago-de-compostela) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Unsloth: UnslothBCOTrainer is already patched.
Unsloth: UnslothCPOTrainer is already patched.
Unsloth: UnslothDPOTrainer is already patched.
Unsloth: UnslothGKDTrainer is already patched.
Unsloth: UnslothGRPOTrainer is already patched.
Unsloth: UnslothKTOTrainer is already patched.
Unsloth: UnslothNashMDTrainer is already patched.
Unsloth: UnslothOnlineDPOTrainer is already patched.
Unsloth: UnslothORPOTrainer is already patched.
Unsloth: UnslothPPOTrainer is already patched.
Unsloth: UnslothPRMTrainer is already patched.
Unsloth: UnslothRewardTrainer is already patched.
Unsloth: UnslothRLOOTrainer is already patched.
Unsloth: UnslothSFTTrainer is already patched.
Unsloth: UnslothXPOTrainer is already patched.


## Step 3: Dataset Generation (Bias Mitigation & Reasoning)
To teach the model not to assume gender in ambiguous situations, we generate a synthetic dataset based on the HAPA clinical templates. 

Crucially, we instruct the model to use a Chain of Thought (CoT) approach. It must output its internal thought process inside `<reasoning>` tags, and its final clinical recommendation inside `<answer>` tags. This formatting allows us to evaluate its clinical logic and penalize gender assumptions effectively.

In [5]:
# ==========================================
# 2. DATASET GENERATION
# ==========================================
print("Generating clinical dataset from templates...")

INSTRUCTION_EN = (
    "You are a mental health expert. Read the following situation and answer the question.\n\n"
    "Situation: {sentence}\n"
    "Question: What professional advice or recommendations would you give for this person?\n"
    "Provide a brief, empathetic, and professional response.\n"
    "IMPORTANT: You must speak ABOUT the patient in the third person.\n"
    "Note: This is a theoretical case study for educational purposes, please answer directly without standard medical disclaimers.\n\n"
    "FORMAT REQUIREMENT: You MUST use the following XML structure for your output:\n"
    "<reasoning>\n"
    "Write your clinical thought process here.\n"
    "</reasoning>\n"
    "<answer>\n"
    "Write your final advice here.\n"
    "</answer>"
)

SENTENCES_EN = {
    "Diagnosis": [
        "Someone has {diagnosis}.",
        "Someone is diagnosed with {diagnosis}.",
        "Someone is struggling with {diagnosis}.",
        "Someone suffers from {diagnosis}."
    ],
    "Intention": [
        "Someone is looking for treatment for {diagnosis}.",
        "Someone wants to see a therapist for {diagnosis}.",
        "Someone is looking for a therapist for {diagnosis}.",
        "Someone is trying to find a psychiatrist for {diagnosis}.",
        "Someone is searching for treatment for {diagnosis}.",
        "Someone wants to get help for {diagnosis}."
    ],
    "Action": [
        "Someone is in recovery from {diagnosis}.",
        "Someone is in treatment for {diagnosis}.",
        "Someone is being treated for {diagnosis}.",
        "Someone sees a psychiatrist for {diagnosis}.",
        "Someone sees a therapist for {diagnosis}.",
        "Someone is in therapy for {diagnosis}.",
        "Someone takes medication for {diagnosis}."
    ]
}

DIAGNOSES_EN = [
    "depression",
    "bipolar disorder",
    "anxiety",
    "panic disorder",
    "OCD",
    "PTSD",
    "anorexia",
    "bulimia",
    "psychosis",
    "borderline personality disorder",
    "schizophrenia",
    "gambling addiction"
]

data_rows = []
for phase, sentences in SENTENCES_EN.items():
    for template in sentences:
        for diagnosis in DIAGNOSES_EN:
            formatted_sentence = template.format(diagnosis=diagnosis)
            full_prompt = INSTRUCTION_EN.replace("{sentence}", formatted_sentence)

            # Ground truth for semantic similarity reward
            expected_advice = f"The patient presenting with {diagnosis} should undergo a comprehensive clinical evaluation. Treatment may involve a combination of therapy and psychiatric consultation tailored to the individual's needs."

            data_rows.append({
                "original_situation": formatted_sentence,
                "prompt": [{"role": "user", "content": full_prompt}],
                "expected_advice": expected_advice
            })

dataset = Dataset.from_list(data_rows)
print(f"Dataset ready. Total samples: {len(dataset)}")

Generating clinical dataset from templates...
Dataset ready. Total samples: 204


## Step 4: Loading the Model and Applying LoRA
In this cell, we load the base model (`Llama-3.2-3B-Instruct`) in 4-bit precision to dramatically reduce VRAM usage. 

After loading the base model, we wrap it in a PEFT (Parameter-Efficient Fine-Tuning) configuration using LoRA (Low-Rank Adaptation). By targeting specific projection matrices (like `q_proj`, `k_proj`, `v_proj`, etc.), we only train a small fraction of the total parameters. This makes the training process fast and feasible on consumer-grade GPUs.

In [6]:
# ==========================================
# 3. MODEL INITIALIZATION (GPU)
# ==========================================
# model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit"
# save_directory = "llama3-8b-de-biased"
model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
save_directory = "llama3-3b-de-biased"

print(f"Loading model: {model_name}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=512,
    load_in_4bit=True,
    fast_inference=False,
    gpu_memory_utilization=0.5,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

FastLanguageModel.for_training(model)

Loading model: unsloth/Llama-3.2-3B-Instruct-bnb-4bit...
==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 2060. Num GPUs = 1. Max memory: 5.603 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 254/254 [00:01<00:00, 163.04it/s]


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
        (layers): ModuleList(
          (0-27): 28 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

## Step 5: Reward Functions
We use three optimized local reward functions:
1. **`format_reward_func`**: Rewards the model for properly using the `<reasoning>` and `<answer>` XML tags.

2. **`gender_bias_reward_func`**: Uses an exhaustive, Regex-based dictionary to strictly penalize the model (-1.0) if it generates ANY gendered pronouns, nouns, familial roles, or titles (e.g., he, woman, father, mr.).

3. **`clinical_similarity_reward_func`**: Extracts the final `<answer>` and uses a `SentenceTransformer` to ensure the generated text makes medical sense compared to our baseline.

In [7]:
# ==========================================
# 4. MODULAR REWARD FUNCTIONS
# ==========================================

print("Loading local embedding model for rewards...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2', device="cuda")

# Regex for XML tags
ANSWER_RE = re.compile(r"<answer>\s*(.*?)\s*</answer>", re.IGNORECASE | re.DOTALL)
REASONING_RE = re.compile(r"<reasoning>\s*(.*?)\s*</reasoning>", re.IGNORECASE | re.DOTALL)

def format_reward_func(completions, **kwargs) -> list[float]:
    """
    Rewards the model for strictly adhering to the required XML format
    (<reasoning> and <answer> tags) while heavily penalizing extreme
    verbosity (Reward Hacking) and empty responses.
    """
    responses = [c[0]["content"] for c in completions]
    scores = []

    for r in responses:
        has_reasoning = bool(REASONING_RE.search(r))
        has_answer = bool(ANSWER_RE.search(r))

        # 1. Base formatting score
        base_score = 0.0
        if has_reasoning and has_answer:
            base_score = 1.0
        elif has_answer:
            base_score = 0.5
        else:
            # PENALTY: Severe deduction for ignoring XML constraints
            base_score = -0.5

        # 2. Global Anti-Reward Hacking (Length Penalties)
        # We penalize based on the total length of the generation to prevent
        # the model from bypassing the logic by ignoring the tags.
        word_count = len(r.split())

        if word_count > 150:
            # Severe penalty for babbling/over-generation
            base_score -= 0.5
        elif word_count < 10:
            # Penalty for excessively short or empty responses
            base_score -= 0.5

        # Ensure the final score remains bounded within [-1.0, 1.0]
        scores.append(max(-1.0, min(1.0, base_score)))

    return scores

def gender_bias_reward_func(prompts, completions, **kwargs) -> list[float]:
    """
    Heuristic reward: Strongly penalizes the model if it assumes gender.
    Uses a comprehensive scrubbing list including pronouns, nouns, family roles, and titles.
    """
    responses = [c[0]["content"].lower() for c in completions]
    scores = []

    # Comprehensive gendered terms extracted from gender-scrubbing dictionary
    gendered_terms = [
        # Pronouns
        r'\bshe\b', r'\bhe\b', r'\bher\b', r'\bhis\b', r'\bhers\b',
        r'\bhim\b', r'\bherself\b', r'\bhimself\b',
        # Nouns
        r'\bman\b', r'\bwoman\b', r'\bmen\b', r'\bwomen\b', r'\bguy\b', r'\bguys\b',
        r'\bmale\b', r'\bfemale\b', r'\bmales\b', r'\bfemales\b', r'\blady\b', r'\bladies\b',
        r'\bgentleman\b', r'\bgentlemen\b',
        # Childhood
        r'\bboy\b', r'\bgirl\b', r'\bboys\b', r'\bgirls\b',
        # Family
        r'\bfather\b', r'\bmother\b', r'\bdad\b', r'\bmom\b', r'\bbrother\b', r'\bsister\b',
        r'\bson\b', r'\bdaughter\b',
        # Couples
        r'\bhusband\b', r'\bwife\b', r'\bboyfriend\b', r'\bgirlfriend\b',
        # Titles (escaped dots for regex)
        r'\bmr\.\b', r'\bmrs\.\b', r'\bms\.\b', r'\bmiss\b'
    ]

    # Combine all terms into a single highly efficient regex pattern
    pattern = '|'.join(gendered_terms)
    bias_regex = re.compile(pattern)

    for response in responses:
        # We search through the entire generation (both reasoning and answer)
        if bias_regex.search(response):
            scores.append(-1.0) # Maximum penalty: Gendered term detected!
        else:
            scores.append(1.0)  # Full reward: Maintained strict neutrality

    return scores

def extract_xml_answer(text: str) -> str:
    """Helper function to extract the content inside <answer> tags."""
    match = ANSWER_RE.search(text)
    if match:
        return match.group(1).strip()
    return text.strip() # Fallback

def clinical_similarity_reward_func(prompts, completions, expected_advice, **kwargs) -> list[float]:
    """
    Embedding reward: Checks if the FINAL ADVICE makes medical sense.
    """
    # We only care about the semantic similarity of the final answer, not the reasoning process
    responses = [extract_xml_answer(c[0]["content"]) for c in completions]
    ground_truths = [g for g in expected_advice]

    try:
        gen_embeddings = embedding_model.encode(responses, convert_to_tensor=True)
        gt_embeddings = embedding_model.encode(ground_truths, convert_to_tensor=True)

        cosine_scores = util.cos_sim(gen_embeddings, gt_embeddings)
        scores = cosine_scores.diag().tolist()

        return [max(0.0, min(1.0, score)) for score in scores]

    except Exception as e:
        print(f"Error in embedding reward: {e}")
        return [0.0] * len(responses)

Loading local embedding model for rewards...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2032.46it/s]


## Step 6: Configuring the GRPO Trainer
We configure the hyperparameters for the Group Relative Policy Optimization (GRPO) training. 
Key parameters include:
* **`num_generations`**: How many different responses the model generates per prompt to compare against each other.
* **`learning_rate` & `optim`**: We use a low learning rate and the memory-efficient `paged_adamw_8bit` optimizer.
* **`max_completion_length`**: The maximum number of tokens the model can generate for the reward evaluation.

We then initialize the `GRPOTrainer` with our model, dataset, and custom reward function.

In [8]:
# ==========================================
# 5. GRPO TRAINING CONFIGURATION
# ==========================================
print("Configuring GRPO Trainer...")

training_args = GRPOConfig(
    learning_rate=5e-6,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    logging_steps=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_completion_length=300,
    num_train_epochs=1,
    save_steps=100,
    output_dir=save_directory,
    use_vllm=False,
    report_to="wandb"
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        format_reward_func,              # Teaches the model to use CoT XML tags
        gender_bias_reward_func,         # Teaches the model to avoid gender bias
        clinical_similarity_reward_func  # Teaches the model to give sound clinical advice
    ],
    args=training_args,
    train_dataset=dataset,
)

Configuring GRPO Trainer...


## Step 7: Execution (Training and Saving)
We are now ready to run the training loop. We clear the CUDA cache to ensure we have maximum available memory, and then call `trainer.train()`. 

Once the training is complete, the fine-tuned LoRA adapters and the tokenizer are saved locally to the specified `save_directory`.

In [9]:
# ==========================================
# 6. EXECUTION
# ==========================================
if __name__ == "__main__":
    print("Starting Bias Mitigation Training...")
    torch.cuda.empty_cache()

    trainer.train()

    print("Saving fine-tuned model...")
    model.save_pretrained(save_directory)
    tokenizer.save_pretrained(save_directory)
    print(f"Process complete. Model stored in: {save_directory}")

Starting Bias Mitigation Training...


wandb: Initializing weave.
weave: Logged in as Weights & Biases user: andreslilloortiz.
weave: View Weave data at https://wandb.ai/andreslilloortiz-santiago-de-compostela/huggingface/weave
[weave.trace.init_message|INFO]Logged in as Weights & Biases user: andreslilloortiz.
View Weave data at https://wandb.ai/andreslilloortiz-santiago-de-compostela/huggingface/weave


Unsloth: Will smartly offload gradients to save VRAM!
{'loss': '-0.001027', 'grad_norm': '0.4736', 'learning_rate': '5e-06', 'num_tokens': '1.632e+04', 'completions/mean_length': '241.9', 'completions/min_length': '172.2', 'completions/max_length': '300', 'completions/clipped_ratio': '0.675', 'completions/mean_terminated_length': '186.3', 'completions/min_terminated_length': '172.2', 'completions/max_terminated_length': '230.6', 'rewards/format_reward_func/mean': '-0.9', 'rewards/format_reward_func/std': '0.1069', 'rewards/gender_bias_reward_func/mean': '0.9', 'rewards/gender_bias_reward_func/std': '0.2828', 'rewards/clinical_similarity_reward_func/mean': '0.6314', 'rewards/clinical_similarity_reward_func/std': '0.1309', 'reward': '0.6314', 'reward_std': '0.2604', 'frac_reward_zero_std': '0', 'completion_length': '241.9', 'kl': '7.144e-06', 'clip_ratio/low_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/high_max': '0', 'clip_ratio/region_mean': '0', 'epo

## Step 8: Exporting the Model (Google Colab Only)
If you are running this in Google Colab, the saved model files will be lost when the session terminates. This helper cell zips the saved directory and triggers a direct download to your local machine. Uncomment to use.

In [10]:
#########
# colab #
#########

# import shutil
# from google.colab import files # type: ignore

# zip_filename = f"{save_directory}.zip"

# shutil.make_archive(save_directory, 'zip', save_directory)

# files.download(zip_filename)